# Lab 05 - Blob indexing with Content Understanding

## Required permissions

- **Your identity:** **Search Service Contributor** + **Search Index Data Contributor** on Search and **Storage Blob Data Contributor** on the source container. Sign in with `az login` in your resource tenant.
- **Search system-assigned managed identity:** **Storage Blob Data Reader** on the container and **Cognitive Services User on the parent Foundry resource** for keyless Content Understanding and embeddings. An embedding-only role is not sufficient for Content Understanding.
- **Resources:** an Azure AI Search service with semantic ranker enabled, a Blob container, and a Foundry resource supporting Content Understanding with an embedding deployment. No extra tenant administrator role is required to run the lab.

**Resource prefix:** set `WORKSHOP_PREFIX` in `.env`, for example `alice01` (3-20 lowercase letters/digits, starting with a letter). Your index, data source, skillset and indexer use `alice01-cu-*`; uploaded blobs use `alice01/content-understanding/`. This keeps the lab separate from notebook 04. Prefixes prevent naming collisions, not enforce access isolation.

**Goal:** extract a two-page PDF as Markdown, chunk it, generate embeddings and retrieve evidence with page ranges. Allow 45-60 minutes. You can run this lab independently of notebook 04.

<img src="../docs/content-understanding-search-pipeline.png" alt="PDF-to-Search pipeline: Blob upload, indexer, Content Understanding, embeddings and provenance, index projections, chunk index, hybrid semantic retrieval and evidence with page ranges" width="1000">

Notebook 04 uses built-in text extraction and Text Split. Here **Content Understanding replaces both extraction and splitting**; there is no second Text Split skill. We use the stable Search REST API `2026-04-01`, not preview semantic chunking or image descriptions. The service returns Markdown content; layout preservation should be inspected, not assumed perfect.

## Why Content Understanding is useful for AI Search

- **More useful document representation:** Markdown can retain headings and table structure rather than flattening everything into plain text. Inspect the sample receipt table to see how well its relationships survive extraction.
- **Retrieval-ready chunks:** one skill extracts and chunks the document, producing `text_sections` that feed directly into embedding and index-projection steps. This reduces the separate extraction and splitting code you need to maintain.
- **Traceable evidence:** location metadata records the source page range and chunk order. Together with the Blob URL, it helps users locate and verify the evidence behind a retrieved result.
- **Fits the Search enrichment pipeline:** the indexer reads the PDF, runs Content Understanding and embeddings, then projects each chunk into its own searchable document. Keyword, vector, hybrid and semantic-ranked queries can use the same indexed content.

This lab uses character-based chunking, not preview semantic chunking. Better document structure and citations do not guarantee correct extraction or answers: inspect the Markdown, page ranges and retrieved evidence.

**Before you begin:** configure your existing resources in `.env` and `.env.azure`, sign in with Azure CLI, and run the C# kernel from `c-sharp-notebooks`. Verify the user and Search-identity permissions above and allow the necessary Blob/Foundry network access. The [deployment guide](../infra/README.md) is one option for provisioning the resources and role assignments, not a prerequisite to using your own resources. The sample PDF is original synthetic content; no extra chat-model deployment, API key, account key or SAS is needed.

## 1. Load configuration

Run the C# kernel from `c-sharp-notebooks`, just like labs 01-04. Settings load from `..\.env`, then `..\.env.azure`; both override existing environment variables, and `.env.azure` wins for duplicates. Keep `WORKSHOP_PREFIX` in your untracked `.env` and configure the tenant, Search, Storage, Foundry account and embedding deployment settings listed in the code below. Rerun this cell after changing settings.

The sample PDF is read from `..\data\search\travel-handbook.pdf`. All objects use separate `-cu-` names and a separate Blob folder, so this lab does not overwrite notebook 04's data or require that notebook to have run.

In [ ]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.IO;
using System.Linq;
using System.Net;
using System.Net.Http;
using System.Net.Http.Headers;
using System.Text;
using System.Text.Json;
using System.Text.Json.Nodes;
using System.Text.RegularExpressions;
using System.Threading;
using System.Threading.Tasks;

// Inline dotenv, validation, token and HTTP helpers: no console project/source imports.
public static class Workshop
{
public static void LoadEnvFile(string path)
    {
        if (!File.Exists(path))
            return;

        foreach (string line in File.ReadAllLines(path))
        {
            string setting = line.Trim();
            if (setting.Length == 0 || setting.StartsWith("#", StringComparison.Ordinal))
                continue;
            if (setting.StartsWith("export ", StringComparison.Ordinal))
                setting = setting.Substring(7).TrimStart();

            int equals = setting.IndexOf('=');
            if (equals <= 0)
                continue;
            string name = setting.Substring(0, equals).Trim();
            string value = setting.Substring(equals + 1).Trim();
            if (value.Length >= 2 &&
                ((value[0] == '"' && value[value.Length - 1] == '"') ||
                 (value[0] == '\'' && value[value.Length - 1] == '\'')))
            {
                value = value.Substring(1, value.Length - 2);
            }
            else
            {
                int comment = value.IndexOf(" #", StringComparison.Ordinal);
                if (comment >= 0)
                    value = value.Substring(0, comment).TrimEnd();
            }
            Environment.SetEnvironmentVariable(name, value);
        }
    }
}
public sealed record SearchSettings(string Root, string Tenant, string SearchEndpoint,
    string StorageEndpoint, string StorageId, string Container, string Account,
    string Deployment, string Model, string Prefix)
{
    public string Index => Prefix + "-cu-chunks";
    public string DataSource => Prefix + "-cu-source";
    public string Skills => Prefix + "-cu-skills";
    public string Indexer => Prefix + "-cu-indexer";
    public string BlobPrefix => Prefix + "/content-understanding/";
    public string FoundryEndpoint => $"https://{Account}.services.ai.azure.com";
    public string EmbeddingEndpoint => $"https://{Account}.openai.azure.com";

    public static SearchSettings Load(string root)
    {
        Workshop.LoadEnvFile(Path.Combine(root, ".env"));
        Workshop.LoadEnvFile(Path.Combine(root, ".env.azure"));
        string Required(string name) => Environment.GetEnvironmentVariable(name)?.Trim() is { Length: > 0 } value
            ? value : throw new InvalidOperationException($"Missing setting: {name}");
        var settings = new SearchSettings(Path.GetFullPath(root), Required("AZURE_TENANT_ID"),
            Required("AZURE_SEARCH_ENDPOINT").TrimEnd('/'), Required("AZURE_STORAGE_BLOB_ENDPOINT").TrimEnd('/'),
            Required("AZURE_STORAGE_RESOURCE_ID"), Required("AZURE_STORAGE_CONTAINER_NAME"),
            Required("AZURE_AI_ACCOUNT_NAME"), Required("AZURE_AI_EMBEDDING_DEPLOYMENT_NAME"),
            Environment.GetEnvironmentVariable("AZURE_AI_EMBEDDING_MODEL_NAME")?.Trim() is { Length: > 0 } model
                ? model : "text-embedding-3-small", Required("WORKSHOP_PREFIX"));
        settings.Validate();
        return settings;
    }

    public void Validate()
    {
        if (!Guid.TryParseExact(Tenant, "D", out _) || !Regex.IsMatch(Prefix, @"^[a-z][a-z0-9]{2,19}$"))
            throw new InvalidOperationException("Use a tenant GUID and a 3-20 lowercase letter/digit prefix starting with a letter.");
        ValidateEndpoint(SearchEndpoint, ".search.windows.net");
        ValidateEndpoint(StorageEndpoint, ".blob.core.windows.net");
        if (!Regex.IsMatch(StorageId, @"^/subscriptions/[0-9a-fA-F-]{36}/resourceGroups/[a-zA-Z0-9_.()-]+/providers/Microsoft\.Storage/storageAccounts/[a-z0-9]{3,24}$") ||
            !Regex.IsMatch(Container, @"^[a-z0-9][a-z0-9-]{1,61}[a-z0-9]$") ||
            !Regex.IsMatch(Account, @"^[a-zA-Z0-9][a-zA-Z0-9-]{1,62}$") ||
            !Regex.IsMatch(Deployment, @"^[a-zA-Z0-9][a-zA-Z0-9_.-]{0,127}$") ||
            !Regex.IsMatch(Model, @"^[a-zA-Z0-9][a-zA-Z0-9_.-]{0,127}$"))
            throw new InvalidOperationException("Invalid Storage resource ID, container, AI account, deployment or model.");
    }

    static void ValidateEndpoint(string value, string suffix)
    {
        if (!Uri.TryCreate(value, UriKind.Absolute, out var uri) || uri.Scheme != "https" ||
            !uri.Host.EndsWith(suffix, StringComparison.OrdinalIgnoreCase) || uri.Host.Length <= suffix.Length ||
            uri.AbsolutePath != "/" || uri.Query != "" || uri.Fragment != "" || uri.UserInfo != "" || !uri.IsDefaultPort)
            throw new InvalidOperationException($"Expected an HTTPS Azure endpoint ending in {suffix}, without a path/query.");
    }
}


string root = Path.GetFullPath("..");
SearchSettings settings = SearchSettings.Load(root);
const int Dimensions = 1536;
const string SemanticConfiguration = "chunk-semantic";
const string apiVersion = "2026-04-01";
const string searchAudience = "https://search.azure.com/";
const string storageAudience = "https://storage.azure.com/";
HttpClient client = new HttpClient { Timeout = TimeSpan.FromSeconds(100) };
Func<string, Task<string>> tokenProvider = resource => AzureCliTokenAsync(settings.Tenant, resource);
Func<TimeSpan, Task> pause = duration => Task.Delay(duration);
Func<DateTimeOffset> clock = () => DateTimeOffset.UtcNow;
string? previousRunStart = null;
bool requestedRunAccepted = false;
async Task<string> AzureCliTokenAsync(string tenant, string resource)
    {
        if (!Guid.TryParseExact(tenant, "D", out _) || resource is not (searchAudience or storageAudience))
            throw new InvalidOperationException("Invalid Azure CLI tenant or audience.");
        var start = new ProcessStartInfo
        {
            FileName = OperatingSystem.IsWindows() ? "cmd.exe" : "az",
            UseShellExecute = false,
            RedirectStandardOutput = true,
            RedirectStandardError = true,
            CreateNoWindow = true,
        };
        if (OperatingSystem.IsWindows())
            foreach (string arg in new[] { "/d", "/c", "az" })
                start.ArgumentList.Add(arg);
        foreach (string arg in new[] { "account", "get-access-token", "--resource", resource, "--tenant", tenant, "--output", "json" })
            start.ArgumentList.Add(arg);
        using var process = Process.Start(start) ?? throw new InvalidOperationException("Could not start Azure CLI.");
        Task<string> output = process.StandardOutput.ReadToEndAsync();
        Task<string> errors = process.StandardError.ReadToEndAsync();
        using var timeout = new CancellationTokenSource(TimeSpan.FromSeconds(60));
        try { await process.WaitForExitAsync(timeout.Token); }
        catch (OperationCanceledException)
        {
            if (!process.HasExited)
                process.Kill(entireProcessTree: true);
            await process.WaitForExitAsync();
            await Task.WhenAll(output, errors);
            throw new InvalidOperationException("Azure CLI authentication timed out. Sign in to the workshop tenant.");
        }
        await errors;
        if (process.ExitCode != 0)
            throw new InvalidOperationException("Azure CLI authentication failed. Run az login --tenant with AZURE_TENANT_ID.");
        return JsonNode.Parse(await output)?["accessToken"]?.GetValue<string>() is { Length: > 0 } token
            ? token : throw new InvalidOperationException("Azure CLI returned no access token.");
    }
void CheckResponse(HttpResponseMessage response, string text, params int[] expected)
    {
        if (expected.Contains((int)response.StatusCode))
            return;
        string message = $"HTTP {(int)response.StatusCode} {response.ReasonPhrase}; expected {string.Join(", ", expected)}.";
        Console.Error.WriteLine(message);
        Console.Error.WriteLine("Response body: " + (text.Length == 0 ? "(empty)" : text[..Math.Min(8000, text.Length)]));
        foreach (string name in new[] { "x-ms-request-id", "request-id", "retry-after", "x-ms-retry-after-ms" })
            if (response.Headers.TryGetValues(name, out var values))
                Console.Error.WriteLine($"{name}: {string.Join(", ", values)}");
        throw new HttpRequestException(message, null, response.StatusCode);
    }

async Task<JsonObject> SendCheckedAsync(HttpRequestMessage request, params int[] expected)
{
    // Requests and Authorization stay visible in the cells; no automatic HTTP retries.
    using var response = await client.SendAsync(request);
    string text = await response.Content.ReadAsStringAsync();
    CheckResponse(response, text, expected);
    if (response.StatusCode == HttpStatusCode.NotFound)
        return new JsonObject { ["missing"] = true };
    return string.IsNullOrWhiteSpace(text) ? new JsonObject() : JsonNode.Parse(text)!.AsObject();
}
Console.WriteLine("Search: " + settings.SearchEndpoint);
Console.WriteLine("Index: " + settings.Index);
Console.WriteLine("Source folder: " + settings.Container + "/" + settings.BlobPrefix);

Console.WriteLine("Foundry enrichment: " + settings.FoundryEndpoint);

void ValidatePageRanges(JsonArray chunks)
{
    // This validation is specific to the original two-page workshop sample.
    foreach (JsonNode? item in chunks)
        if (item?["page_from"] is not JsonValue from || !from.TryGetValue<int>(out int first) ||
            item?["page_to"] is not JsonValue to || !to.TryGetValue<int>(out int last) ||
            first < 1 || first > last || last > 2)
            throw new InvalidOperationException("Expected valid page ranges from the two-page sample; inspect locationMetadata.");
}


## 2. Upload the original PDF

Open [`travel-handbook.pdf`](../data/search/travel-handbook.pdf) first. Page 1 includes a receipt-requirements table; page 2 includes exceptions and an example. This is a fictional policy, not advice or a real employer policy. The upload uses `application/pdf`, Entra authentication and your own folder. Repeating it overwrites only this sample blob.

In [ ]:
// Upload only the original synthetic PDF; never push documents directly to Search.
string[] sampleNames = { "travel-handbook.pdf" };
foreach (string name in sampleNames)
    if (!File.Exists(Path.Combine(root, "data", "search", name)))
        throw new FileNotFoundException("Expected workshop sample: " + name);
foreach (string name in sampleNames)
{
    string blobName = settings.BlobPrefix + name;
    // Escape each path segment while retaining the virtual-folder slashes.
    string storagePath = string.Join("/", blobName.Split('/').Select(Uri.EscapeDataString));
    using var request = new HttpRequestMessage(HttpMethod.Put,
        $"{settings.StorageEndpoint}/{Uri.EscapeDataString(settings.Container)}/{storagePath}");
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(storageAudience));
    request.Headers.Add("x-ms-version", "2023-11-03");
    request.Headers.Add("x-ms-date", DateTimeOffset.UtcNow.ToString("R", CultureInfo.InvariantCulture));
    request.Headers.Add("x-ms-blob-type", "BlockBlob");
    byte[] pdfData = await File.ReadAllBytesAsync(Path.Combine(root, "data", "search", name));
    if (!pdfData.AsSpan().StartsWith("%PDF-"u8))
        throw new InvalidOperationException("Expected a PDF file: " + name);
    request.Content = new ByteArrayContent(pdfData);
    request.Content.Headers.ContentType = MediaTypeHeaderValue.Parse("application/pdf");
    await SendCheckedAsync(request, 201);
    Console.WriteLine("Uploaded: " + blobName);
}

## 3. Create the Blob data source

The `ResourceId` connection uses Search's system-assigned managed identity, not your notebook token. The virtual folder limits which files are selected, but is not a participant security boundary. Do not skip this cell: an indexer cannot reference a data source that does not exist. Source deletion detection is deliberately not configured; deleting a blob alone does not remove previously indexed chunks.

In [ ]:
// ResourceId selects Search's identity, not a key/SAS. query scopes selection, not authorization.
JsonObject dataSource = JsonNode.Parse($$"""
        {
          "name": "{{settings.DataSource}}",
          "type": "azureblob",
          "credentials": {
            "connectionString": "ResourceId={{settings.StorageId}};"
          },
          "container": {
            "name": "{{settings.Container}}",
            "query": "{{settings.BlobPrefix}}"
          }
        }
        """)!.AsObject();
using (var request = new HttpRequestMessage(HttpMethod.Put,
    $"{settings.SearchEndpoint}/datasources/{settings.DataSource}?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    request.Content = new StringContent(dataSource.ToJsonString(), Encoding.UTF8, "application/json");
    var responseBody = await SendCheckedAsync(request, 200, 201, 204);
}
Console.WriteLine("Ready: " + settings.DataSource);

## 4. Define the chunk index

Search projections generate `chunk_id` and `parent_id`; do not map these manually. Each chunk stores Markdown, a 1,536-dimensional embedding, filename/Blob URL and Content Understanding location metadata. `page_from` and `page_to` form a **range**, because a chunk may span pages. `ordinal_position` is ordering metadata, not a page number.

In [ ]:
// Generated chunk/parent IDs, cosine HNSW and query-time vectorizer; skill/model/dimensions agree.
JsonObject index = JsonNode.Parse($$"""
        {
          "name": "{{settings.Index}}",
          "fields": [
            {
              "name": "chunk_id",
              "type": "Edm.String",
              "key": true,
              "searchable": true,
              "analyzer": "keyword"
            },
            {
              "name": "parent_id",
              "type": "Edm.String",
              "filterable": true
            },
            {
              "name": "chunk",
              "type": "Edm.String",
              "searchable": true
            },
            {
              "name": "title",
              "type": "Edm.String",
              "searchable": true
            },
            {
              "name": "page_from",
              "type": "Edm.Int32",
              "filterable": true
            },
            {
              "name": "page_to",
              "type": "Edm.Int32",
              "filterable": true
            },
            {
              "name": "ordinal_position",
              "type": "Edm.Int32",
              "sortable": true
            },
            {
              "name": "location_source",
              "type": "Edm.String"
            },
            {
              "name": "text_vector",
              "type": "Collection(Edm.Single)",
              "searchable": true,
              "retrievable": true,
              "dimensions": {{Dimensions}},
              "vectorSearchProfile": "chunk-vector-profile"
            },
            {
              "name": "source",
              "type": "Edm.ComplexType",
              "fields": [
                {
                  "name": "file_name",
                  "type": "Edm.String",
                  "searchable": true
                },
                {
                  "name": "url",
                  "type": "Edm.String"
                }
              ]
            }
          ],
          "vectorSearch": {
            "algorithms": [
              {
                "name": "chunk-hnsw",
                "kind": "hnsw",
                "hnswParameters": {
                  "metric": "cosine"
                }
              }
            ],
            "profiles": [
              {
                "name": "chunk-vector-profile",
                "algorithm": "chunk-hnsw",
                "vectorizer": "chunk-vectorizer"
              }
            ],
            "vectorizers": [
              {
                "name": "chunk-vectorizer",
                "kind": "azureOpenAI",
                "azureOpenAIParameters": {
                  "resourceUri": "{{settings.EmbeddingEndpoint}}",
                  "deploymentId": "{{settings.Deployment}}",
                  "modelName": "{{settings.Model}}"
                }
              }
            ]
          },
          "semantic": {
            "defaultConfiguration": "{{SemanticConfiguration}}",
            "configurations": [
              {
                "name": "{{SemanticConfiguration}}",
                "prioritizedFields": {
                  "titleField": {
                    "fieldName": "title"
                  },
                  "prioritizedContentFields": [
                    {
                      "fieldName": "chunk"
                    }
                  ]
                }
              }
            ]
          }
        }
        """)!.AsObject();
using (var request = new HttpRequestMessage(HttpMethod.Put,
    $"{settings.SearchEndpoint}/indexes/{settings.Index}?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    request.Content = new StringContent(index.ToJsonString(), Encoding.UTF8, "application/json");
    var responseBody = await SendCheckedAsync(request, 200, 201, 204);
}
Console.WriteLine("Ready: " + settings.Index);

## 5. Extract, chunk and embed with managed identities

`file_data` provides the original PDF. Content Understanding produces `/document/text_sections/*` objects whose `content` is Markdown. The embedding skill reads each object's **content**, not the whole object. Shaper preserves the original filename and URL.

`cognitiveServices` attaches the Foundry resource for keyless analysis and billing. `identity=null` selects Search's system identity. The same identity calls embeddings when `apiKey` and `authIdentity` are omitted. Fixed-size chunking uses 1,000 characters and 200-character overlap; preview semantic chunking and image descriptions are intentionally not enabled.

See the [Content Understanding skill parameters](https://learn.microsoft.com/azure/search/cognitive-search-skill-content-understanding#skill-parameters) for chunking options and defaults.

In [ ]:
// Original PDF -> Markdown sections -> content embeddings; null identity selects Search system MI.
// pages means size-based character chunks, not PDF pages. No OCR or cognitiveServices attachment.
// Search's system identity embeds each chunk; Shaper preserves provenance; projections generate IDs.
JsonObject skillset = JsonNode.Parse($$"""
        {
          "name": "{{settings.Skills}}",
          "description": "Content Understanding PDF extraction, chunking, vectors and page provenance.",
          "cognitiveServices": {
            "@odata.type": "#Microsoft.Azure.Search.AIServicesByIdentity",
            "subdomainUrl": "{{settings.FoundryEndpoint}}",
            "identity": null
          },
          "skills": [
            {
              "@odata.type": "#Microsoft.Skills.Util.ContentUnderstandingSkill",
              "name": "understand-pdf",
              "context": "/document",
              "extractionOptions": [
                "locationMetadata"
              ],
              "chunkingProperties": {
                "unit": "characters",
                "maximumLength": 1000,
                "overlapLength": 200
              },
              "inputs": [
                {
                  "name": "file_data",
                  "source": "/document/file_data"
                }
              ],
              "outputs": [
                {
                  "name": "text_sections",
                  "targetName": "text_sections"
                }
              ]
            },
            {
              "@odata.type": "#Microsoft.Skills.Text.AzureOpenAIEmbeddingSkill",
              "name": "embed-sections",
              "context": "/document/text_sections/*",
              "resourceUri": "{{settings.EmbeddingEndpoint}}",
              "deploymentId": "{{settings.Deployment}}",
              "modelName": "{{settings.Model}}",
              "dimensions": {{Dimensions}},
              "inputs": [
                {
                  "name": "text",
                  "source": "/document/text_sections/*/content"
                }
              ],
              "outputs": [
                {
                  "name": "embedding",
                  "targetName": "text_vector"
                }
              ]
            },
            {
              "@odata.type": "#Microsoft.Skills.Util.ShaperSkill",
              "name": "shape-provenance",
              "context": "/document",
              "inputs": [
                {
                  "name": "file_name",
                  "source": "/document/metadata_storage_name"
                },
                {
                  "name": "url",
                  "source": "/document/metadata_storage_path"
                }
              ],
              "outputs": [
                {
                  "name": "output",
                  "targetName": "provenance"
                }
              ]
            }
          ],
          "indexProjections": {
            "selectors": [
              {
                "targetIndexName": "{{settings.Index}}",
                "parentKeyFieldName": "parent_id",
                "sourceContext": "/document/text_sections/*",
                "mappings": [
                  {
                    "name": "chunk",
                    "source": "/document/text_sections/*/content"
                  },
                  {
                    "name": "text_vector",
                    "source": "/document/text_sections/*/text_vector"
                  },
                  {
                    "name": "title",
                    "source": "/document/metadata_storage_name"
                  },
                  {
                    "name": "source",
                    "source": "/document/provenance"
                  },
                  {
                    "name": "page_from",
                    "source": "/document/text_sections/*/locationMetadata/pageNumberFrom"
                  },
                  {
                    "name": "page_to",
                    "source": "/document/text_sections/*/locationMetadata/pageNumberTo"
                  },
                  {
                    "name": "ordinal_position",
                    "source": "/document/text_sections/*/locationMetadata/ordinalPosition"
                  },
                  {
                    "name": "location_source",
                    "source": "/document/text_sections/*/locationMetadata/source"
                  }
                ]
              }
            ],
            "parameters": {
              "projectionMode": "skipIndexingParentDocuments"
            }
          }
        }
        """)!.AsObject();
using (var request = new HttpRequestMessage(HttpMethod.Put,
    $"{settings.SearchEndpoint}/skillsets/{settings.Skills}?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    request.Content = new StringContent(skillset.ToJsonString(), Encoding.UTF8, "application/json");
    var responseBody = await SendCheckedAsync(request, 200, 201, 204);
}
Console.WriteLine("Ready: " + settings.Skills);

## 6. Run the Blob indexer

**`allowSkillsetToReadFileData=true` is essential:** it exposes `/document/file_data` to Content Understanding. `dataToExtract='allMetadata'` leaves content extraction to the skill while retaining Blob metadata. The indexer selects PDFs only. Creation starts a run; updating an existing idle indexer explicitly starts a new one. Zero failed-item tolerance prevents partial indexing from being reported as success.

In [ ]:
// Clear acceptance first: a failed rerun must not reuse an earlier successful request.
requestedRunAccepted = false;
previousRunStart = null;
JsonObject indexerStatus = new JsonObject();
using (var request = new HttpRequestMessage(HttpMethod.Get,
    $"{settings.SearchEndpoint}/indexers/{settings.Indexer}/status?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    indexerStatus = await SendCheckedAsync(request, 200, 404);
}
bool indexerExists = indexerStatus["missing"]?.GetValue<bool>() != true;
JsonNode? previousRun = indexerStatus["lastResult"];
if (previousRun?["status"]?.GetValue<string>() == "inProgress")
    throw new InvalidOperationException("The indexer is already running. Wait before updating it.");
previousRunStart = previousRun?["startTime"]?.GetValue<string>();
JsonObject indexer = JsonNode.Parse($$"""
        {
          "name": "{{settings.Indexer}}",
          "dataSourceName": "{{settings.DataSource}}",
          "targetIndexName": "{{settings.Index}}",
          "skillsetName": "{{settings.Skills}}",
          "parameters": {
            "maxFailedItems": 0,
            "maxFailedItemsPerBatch": 0,
            "configuration": {
              "dataToExtract": "allMetadata",
              "allowSkillsetToReadFileData": true,
              "indexedFileNameExtensions": ".pdf"
            }
          }
        }
        """)!.AsObject();
using (var request = new HttpRequestMessage(HttpMethod.Put,
    $"{settings.SearchEndpoint}/indexers/{settings.Indexer}?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    request.Content = new StringContent(indexer.ToJsonString(), Encoding.UTF8, "application/json");
    var responseBody = await SendCheckedAsync(request, 200, 201, 204);
}
if (indexerExists)
{
    // Updating an idle existing indexer does not substitute for explicitly requesting its run.
using (var request = new HttpRequestMessage(HttpMethod.Post,
    $"{settings.SearchEndpoint}/indexers/{settings.Indexer}/run?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    var responseBody = await SendCheckedAsync(request, 202);
}
    Console.WriteLine("New run requested for the existing indexer.");
}
// A new indexer starts automatically. No schedule is configured.
requestedRunAccepted = true;
Console.WriteLine("Indexer ready: " + settings.Indexer);


In [ ]:
if (!requestedRunAccepted)
    throw new InvalidOperationException("Run the indexer cell first; no accepted request is pending.");
DateTimeOffset deadline = clock().AddMinutes(15);
bool completed = false;
for (int attempt = 0; attempt < 90 && clock() < deadline; attempt++)
{
    JsonObject status = new JsonObject();
using (var request = new HttpRequestMessage(HttpMethod.Get,
    $"{settings.SearchEndpoint}/indexers/{settings.Indexer}/status?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    status = await SendCheckedAsync(request, 200);
}
    JsonNode? lastRun = status["lastResult"];
    string? state = lastRun?["status"]?.GetValue<string>();
    string? start = lastRun?["startTime"]?.GetValue<string>();
    Console.WriteLine($"Run: {state} | processed: {lastRun?["itemsProcessed"]} | failed: {lastRun?["itemsFailed"]}");
    // While queued, the previous success can still be visible. It is not this requested run.
    bool isRequestedRun = !string.IsNullOrEmpty(start) && start != previousRunStart;
    if (status["status"]?.GetValue<string>() == "error" ||
        (isRequestedRun && state is "transientFailure" or "persistentFailure" or "error"))
        throw new InvalidOperationException(status.ToJsonString());
    if (isRequestedRun && state == "success")
    {
        if ((lastRun?["itemsFailed"]?.GetValue<int>() ?? 0) != 0 || lastRun?["errors"] is JsonArray { Count: > 0 })
            throw new InvalidOperationException(lastRun!.ToJsonString());
        if (clock() >= deadline) break;
        Console.WriteLine("Warnings: " + (lastRun?["warnings"]?.ToJsonString() ?? "[]"));
        completed = true;
        requestedRunAccepted = false;
        break;
    }
    if (attempt < 89 && clock() < deadline)
        await pause(TimeSpan.FromSeconds(10));
}
if (!completed)
    throw new TimeoutException("Indexer did not complete within 15 minutes. Inspect its status before retrying.");


## 7. Inspect Markdown, page ranges and provenance

Expect several chunks from the one PDF. Compare the receipt table with page 1 and the missing-receipt rules with page 2. A range such as 1-2 means a chunk spans both pages; do not label it as a single page. Inspect the PDF to verify that extracted text and table structure are accurate. The Blob URL still requires authenticated access.

You can also open your index's **Search explorer** in Azure AI Search to inspect the indexed chunks directly. In this example, the returned document includes table content in `chunk`, the source filename in `title`, `page_from=1` and `page_to=2`, ordering/location metadata, and the stored `text_vector`. Compare these fields with the notebook results and original PDF; the screenshot is illustrative, not a required chunk count or ranking.

<img src="../docs/content-understanding-search-explorer.png" alt="Azure AI Search Search Explorer showing extracted table content, source PDF, page range, location metadata and stored embedding" width="900">

In [ ]:
JsonObject inspectBody = JsonNode.Parse($$"""
        {
          "search": "*",
          "count": true,
          "top": 100,
          "orderby": "ordinal_position",
          "select": "chunk_id,parent_id,title,chunk,source,page_from,page_to,ordinal_position"
        }
        """)!.AsObject();
JsonObject chunksResult = new JsonObject();
using (var request = new HttpRequestMessage(HttpMethod.Post,
    $"{settings.SearchEndpoint}/indexes/{settings.Index}/docs/search?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    request.Content = new StringContent(inspectBody.ToJsonString(), Encoding.UTF8, "application/json");
    chunksResult = await SendCheckedAsync(request, 200);
}
if (chunksResult["value"] is not JsonArray { Count: > 0 })
    throw new InvalidOperationException("No chunks were indexed. Inspect indexer errors and the Blob folder prefix.");
ValidatePageRanges(chunksResult["value"]!.AsArray());
Console.WriteLine("Total chunks: " + chunksResult["@odata.count"]);
Console.WriteLine(chunksResult["value"]!.ToJsonString(new JsonSerializerOptions { WriteIndented = true }));


## 8. Retrieve evidence with hybrid search and semantic ranking

Search generates the query embedding using the index's Azure OpenAI vectorizer. Hybrid retrieval combines keyword and vector candidates; semantic ranking reranks them and produces extractive captions, not an LLM-written answer. Explain the receipt requirements using retrieved text and cite the filename and page range. Check every claim against the PDF.

In [ ]:
string question = "Which travel expenses always need itemized receipts, and what if a hotel receipt is missing?";
if (string.IsNullOrWhiteSpace(question))
    throw new ArgumentException("Provide a non-empty semantic-hybrid question.");
JsonObject semanticHybridBody = new JsonObject
{
    ["search"] = question, ["searchFields"] = "chunk",
    ["vectorQueries"] = new JsonArray(new JsonObject
    {
        ["kind"] = "text", ["text"] = question, ["fields"] = "text_vector", ["k"] = 50
    }),
    ["queryType"] = "semantic", ["semanticConfiguration"] = SemanticConfiguration, ["captions"] = "extractive",
    ["select"] = "chunk_id,title,chunk,source,page_from,page_to", ["top"] = 3
};
JsonObject semanticHybridResult = new JsonObject();
using (var request = new HttpRequestMessage(HttpMethod.Post,
    $"{settings.SearchEndpoint}/indexes/{settings.Index}/docs/search?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    request.Content = new StringContent(semanticHybridBody.ToJsonString(), Encoding.UTF8, "application/json");
    semanticHybridResult = await SendCheckedAsync(request, 200);
}
if (semanticHybridResult["@search.semanticPartialResponseReason"] is not null)
    throw new InvalidOperationException("Semantic ranking did not complete: " + semanticHybridResult.ToJsonString());
if (semanticHybridResult["value"] is not JsonArray { Count: > 0 })
    throw new InvalidOperationException("No evidence returned. Inspect the indexed chunks before continuing.");
Console.WriteLine(semanticHybridResult.ToJsonString(new JsonSerializerOptions { WriteIndented = true }));

## References and next lab

- [Content Understanding skill](https://learn.microsoft.com/azure/search/cognitive-search-skill-content-understanding)
- [Keyless skillset attachment and billing](https://learn.microsoft.com/azure/search/cognitive-search-attach-cognitive-services)
- [Content Understanding regions](https://learn.microsoft.com/azure/ai-services/content-understanding/language-region-support)
- [Index projections](https://learn.microsoft.com/azure/search/search-how-to-define-index-projections)
- [Azure OpenAI embedding skill](https://learn.microsoft.com/azure/search/cognitive-search-skill-azure-openai-embedding)

Next: [Python notebook 06](../python-notebooks/06-push-chunks-vectors.ipynb) chunks TXT files and generates embeddings in Python, then pushes them to a separate index without a Blob indexer.
